#Initilization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

# Reading from bronze


In [0]:
df = spark.table("databricks_lakehouse.bronze.crm_cust_info") # same as spark.read.table
display(df)

#Data transformation

#### Trim all the string elements


In [0]:

'''for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.trim(F.col(field.name))) 

When you use a single .select() or .withColumns(), Apache Spark looks at all the transformations simultaneously and optimizes the physical execution path on your cluster.
When we hide .withColumn inside a Python loop, you are forcing Spark to evaluate each column sequentially, blind to the next step, which drastically destroys processing speed.'''

#### Now we do the more optimized way of doing the above

In [0]:
# we can use withColumns instead of withColumn. It is more optimized and takes a dict as object

trim_plan = {
    field.name: F.trim(F.col(field.name))
    for field in df.schema.fields
    if isinstance(field.dataType, StringType)
            }

df = df.withColumns(trim_plan)
display(df)

#### Converting abbreviation to full form eg- M - > married S - > single

In [0]:
df = (
    df.withColumn(
        "cst_marital_status", F.when(F.upper(F.col("cst_marital_status"))=="M", "Married")
        .when(F.upper(F.col('cst_marital_status'))=='S', "Single")
        .otherwise("N/A")
        )
    
    .withColumn(
        "cst_gndr", F.when(F.upper(F.col("cst_gndr"))=="M", "Male")
        .when(F.upper(F.col("cst_gndr"))=="F", "Female")
        .otherwise("N/A")
        )
    )
    
df.display()


####removing rows with no cust_id

In [0]:
df = df.filter(F.col("cst_id").isNotNull())

#### Removing nulls in the name columns

In [0]:
df = (
    df.withColumn("cst_firstname",F.coalesce(F.col("cst_firstname"), F.lit("N/A")))
    .withColumn("cst_lastname",F.coalesce(F.col("cst_lastname"), F.lit("N/A")))
    )
df.display()


####Last null checks before renaming the columns

In [0]:
display(df.filter(
    (F.col("cst_id").isNull()) |
    (F.col("cst_key").isNull()) |
    (F.col("cst_firstname").isNull()) |
    (F.col("cst_lastname").isNull()) |
    (F.col("cst_marital_status").isNull()) |
    (F.col("cst_gndr").isNull()) |
    (F.col("cst_create_date").isNull())
))

#### Renameing the columns appropriately

In [0]:
rename_map = {          #in case of large number of columns, we can use a dict comprehension but now this is enough
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "customer_firstname",
    "cst_lastname": "customer_lastname",
    "cst_marital_status": "customer_marital_status",
    "cst_gndr": "customer_gender",
    "cst_create_date": "customer_created_date"
    }

df = df.withColumnsRenamed(rename_map)  #same as withColumns with an s it is more optimized and takes dict.
df.display()

#Write into Silver Table

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("databricks_lakehouse.silver.crm_customers")
)

#checking the table

In [0]:
%sql
select * from databricks_lakehouse.silver.crm_customers limit 10